# STEP-2

In [ ]:
from pathlib import Path
import pandas as pd


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = PROJECT_ROOT / "data" / "raw" / "banking_loan_approval_dataset_100k.csv"


df = pd.read_csv(RAW_FILE, encoding="utf-8-sig")

# Clean column headers
df.columns = df.columns.str.strip()

print("SC09 BANKING LOAN APPROVAL DATA INSPECTION")
print("-" * 45)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())

SC09 BANKING LOAN APPROVAL DATA INSPECTION
---------------------------------------------
Dataset shape: 100000 rows, 17 columns

Column names:
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing_loan', 'personal_loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'LOAN_APPROVAL']


In [30]:
# 1. Independent working copy
clean_df = df.copy()

# 2. Target variable for SC09
target_col = "LOAN_APPROVAL"

# 3. Define feature columns (all columns except the target)
feature_cols = [col for col in clean_df.columns if col != target_col]

print("--- FEATURE COLUMN LIST ---")
print(feature_cols)

print("\n--- INDEPENDENCE VERIFICATION ---")
print(f"clean_df is a distinct object from df: {clean_df is not df}")

--- FEATURE COLUMN LIST ---
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing_loan', 'personal_loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome']

--- INDEPENDENCE VERIFICATION ---
clean_df is a distinct object from df: True


In [31]:
missing_before = clean_df.isnull().sum()
duplicates_before = int(clean_df.duplicated().sum())

print("--- MISSING VALUES BEFORE ---")
print(missing_before)
print(f"\nTotal missing values: {missing_before.sum()}")

print("\n--- DUPLICATES BEFORE ---")
print(f"Total duplicate rows: {duplicates_before}")

--- MISSING VALUES BEFORE ---
age              0
job              0
marital          0
education        0
default          0
balance          0
housing_loan     0
personal_loan    0
contact          0
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome         0
LOAN_APPROVAL    0
dtype: int64

Total missing values: 0

--- DUPLICATES BEFORE ---
Total duplicate rows: 0


In [32]:
# Median impute only numeric feature columns (leaves target untouched)
numeric_features = clean_df[feature_cols].select_dtypes(include="number").columns
clean_df[numeric_features] = clean_df[numeric_features].fillna(clean_df[numeric_features].median())

# Drop duplicate rows and reset index
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

In [33]:
rows_before = len(df)
rows_after = len(clean_df)
duplicates_removed = duplicates_before

print("=" * 35)
print("     DATA CLEANING SUMMARY")
print("=" * 35)
print(f"Rows before cleaning : {rows_before:,}")
print(f"Rows after cleaning  : {rows_after:,}")
print(f"Duplicates removed   : {duplicates_removed:,}")
print("=" * 35)

print("\nMissing values per column after imputation:")
print(clean_df.isna().sum())

     DATA CLEANING SUMMARY
Rows before cleaning : 100,000
Rows after cleaning  : 100,000
Duplicates removed   : 0

Missing values per column after imputation:
age              0
job              0
marital          0
education        0
default          0
balance          0
housing_loan     0
personal_loan    0
contact          0
day              0
month            0
duration         0
campaign         0
pdays            0
previous         0
poutcome         0
LOAN_APPROVAL    0
dtype: int64


In [34]:
# SC09 required columns
required_columns = [
    "age", "job", "marital", "education", "default", "balance",
    "housing_loan", "personal_loan", "contact", "day", "month",
    "duration", "campaign", "pdays", "previous", "poutcome",
    "LOAN_APPROVAL"
]

numeric_columns = clean_df[required_columns].select_dtypes(include="number").columns.tolist()
categorical_columns = [col for col in required_columns if col not in numeric_columns]

print("=" * 50)
print("SC09 PROJECT COLUMN SPECIFICATIONS")
print("=" * 50)
print(f"\nTotal Required Columns ({len(required_columns)}):")
print(required_columns)

print(f"\nNumeric Columns ({len(numeric_columns)}):")
print(numeric_columns)

print(f"\nCategorical Columns ({len(categorical_columns)}):")
print(categorical_columns)

assert set(required_columns).issubset(set(clean_df.columns)), "Missing required columns!"
print("\nVerification Passed: All required banking columns exist.")

SC09 PROJECT COLUMN SPECIFICATIONS

Total Required Columns (17):
['age', 'job', 'marital', 'education', 'default', 'balance', 'housing_loan', 'personal_loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'LOAN_APPROVAL']

Numeric Columns (11):
['age', 'default', 'balance', 'housing_loan', 'personal_loan', 'day', 'duration', 'campaign', 'pdays', 'previous', 'LOAN_APPROVAL']

Categorical Columns (6):
['job', 'marital', 'education', 'contact', 'month', 'poutcome']

Verification Passed: All required banking columns exist.


In [35]:
print("=" * 45)
print("       CLEANED DATA VALIDATION")
print("=" * 45)

# 1. Missing required columns
missing_cols = set(required_columns) - set(clean_df.columns)
print(f"Missing required columns : {len(missing_cols)} {f'({missing_cols})' if missing_cols else '✓'}")

# 2. Remaining missing values
total_nans = int(clean_df.isna().sum().sum())
print(f"Remaining missing values : {total_nans} {'✓' if total_nans == 0 else '✗'}")

# 3. Domain range checks (Age bounds)
invalid_age = int(((clean_df["age"] < 18) | (clean_df["age"] > 120)).sum())
print(f"Out-of-range age values  : {invalid_age} {'✓' if invalid_age == 0 else '✗'}")

# 4. Strictly non-negative feature checks
non_neg_cols = ["age", "duration", "campaign", "previous"]
negative_counts = sum((clean_df[col] < 0).sum() for col in non_neg_cols if col in clean_df.columns)
print(f"Negative measurements    : {negative_counts} {'✓' if negative_counts == 0 else '✗'}")

# 5. Invalid target labels for LOAN_APPROVAL
valid_labels = {0, 1} if pd.api.types.is_numeric_dtype(clean_df["LOAN_APPROVAL"]) else {"yes", "no", "0", "1"}
invalid_targets = int((~clean_df["LOAN_APPROVAL"].isin(valid_labels)).sum())
print(f"Invalid target labels    : {invalid_targets} {'✓' if invalid_targets == 0 else '✗'}")

# 6. Duplicate rows
duplicate_count = int(clean_df.duplicated().sum())
print(f"Duplicate rows           : {duplicate_count} {'✓' if duplicate_count == 0 else '✗'}")

print("=" * 45)
print("Validation complete.")

       CLEANED DATA VALIDATION
Missing required columns : 0 ✓
Remaining missing values : 0 ✓
Out-of-range age values  : 0 ✓
Negative measurements    : 0 ✓
Invalid target labels    : 0 ✓
Duplicate rows           : 0 ✓
Validation complete.


In [36]:
# Aggregate validation flags from the checks
validation_errors = (
    len(missing_cols)
    + total_nans
    + invalid_age
    + negative_counts
    + invalid_targets
    + duplicate_count
)

if validation_errors == 0:
    print("=" * 45)
    print("VALIDATION PASSED")
    print("=" * 45)
    print("Cleaned data is ready for splitting.")
else:
    print("=" * 45)
    print("VALIDATION FAILED")
    print("=" * 45)
    print(f"Total validation issues found: {validation_errors}")
    print("Please fix the remaining issues before splitting.")

VALIDATION PASSED
Cleaned data is ready for splitting.


In [37]:
print("=" * 45)
print("       DATA TYPES SPECIFICATION")
print("=" * 45)
print(clean_df.dtypes)

print("\n" + "=" * 45)
print("  TARGET CLASS DISTRIBUTION (LOAN_APPROVAL)")
print("=" * 45)
# Value counts and normalized percentages for the SC09 target
target_counts = clean_df["LOAN_APPROVAL"].value_counts().sort_index()
target_pcts = clean_df["LOAN_APPROVAL"].value_counts(normalize=True).sort_index() * 100

summary_df = pd.DataFrame({"Count": target_counts, "Percentage (%)": target_pcts.round(2)})
print(summary_df)

       DATA TYPES SPECIFICATION
age              int64
job                str
marital            str
education          str
default          int64
balance          int64
housing_loan     int64
personal_loan    int64
contact            str
day              int64
month              str
duration         int64
campaign         int64
pdays            int64
previous         int64
poutcome           str
LOAN_APPROVAL    int64
dtype: object

  TARGET CLASS DISTRIBUTION (LOAN_APPROVAL)
               Count  Percentage (%)
LOAN_APPROVAL                       
0              27058           27.06
1              72942           72.94


In [38]:
import numpy as np

# 1. Set explicit fixed random seed
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


# 2. Define stratified split function that shuffles each class separately
def stratified_split(
    df: pd.DataFrame,
    target_col: str = "LOAN_APPROVAL",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = RANDOM_SEED,
):
    """Shuffles each target class independently to preserve exact class

    distribution across train, validation, and test splits.
    """
    assert (
        np.isclose(train_ratio + val_ratio + test_ratio, 1.0)
    ), "Split ratios must sum to 1.0"

    train_chunks = []
    val_chunks = []
    test_chunks = []

    # Process each class separately to ensure stratification
    for class_label in df[target_col].unique():
        class_subset = df[df[target_col] == class_label].sample(
            frac=1.0, random_state=seed
        )
        total_len = len(class_subset)

        train_end = int(train_ratio * total_len)
        val_end = train_end + int(val_ratio * total_len)

        train_chunks.append(class_subset.iloc[:train_end])
        val_chunks.append(class_subset.iloc[train_end:val_end])
        test_chunks.append(class_subset.iloc[val_end:])

    # Concatenate and shuffle the combined splits
    train_df = (
        pd.concat(train_chunks)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )
    val_df = (
        pd.concat(val_chunks)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )
    test_df = (
        pd.concat(test_chunks)
        .sample(frac=1.0, random_state=seed)
        .reset_index(drop=True)
    )

    return train_df, val_df, test_df


print(f"Fixed Random Seed : {RANDOM_SEED}")
print("Split Function     : stratified_split() defined successfully.")

Fixed Random Seed : 42
Split Function     : stratified_split() defined successfully.


In [39]:
# 1. Create data/processed directory if it does not exist
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

# 2. Execute stratified split
train_df, val_df, test_df = stratified_split(
    clean_df, 
    target_col="LOAN_APPROVAL", 
    train_ratio=0.70, 
    val_ratio=0.15, 
    test_ratio=0.15, 
    seed=RANDOM_SEED
)

# 3. Save datasets to data/processed/
clean_df.to_csv(processed_dir / "cleaned_data.csv", index=False)
train_df.to_csv(processed_dir / "train.csv", index=False)
val_df.to_csv(processed_dir / "validation.csv", index=False)
test_df.to_csv(processed_dir / "test.csv", index=False)

# 4. Print size summary and random seed
print("=" * 45)
print("       DATASET SPLIT & EXPORT SUMMARY")
print("=" * 45)
print(f"Random Seed           : {RANDOM_SEED}")
print(f"Cleaned Full Dataset  : {len(clean_df):,} rows")
print(f"Training Set (70%)    : {len(train_df):,} rows")
print(f"Validation Set (15%)  : {len(val_df):,} rows")
print(f"Test Set (15%)        : {len(test_df):,} rows")
print("-" * 45)
print(f"Total Rows Verified   : {len(train_df) + len(val_df) + len(test_df):,} rows")
print(f"Saved Files Location  : {processed_dir}")
print("=" * 45)

       DATASET SPLIT & EXPORT SUMMARY
Random Seed           : 42
Cleaned Full Dataset  : 100,000 rows
Training Set (70%)    : 69,999 rows
Validation Set (15%)  : 14,999 rows
Test Set (15%)        : 15,002 rows
---------------------------------------------
Total Rows Verified   : 100,000 rows
Saved Files Location  : c:\Users\ayush\OneDrive\Desktop\Project26-27\AHPV_Credit_Risk_Detection_Model\data\processed


In [40]:
# 1. Function to display target distribution for a split
def print_target_distribution(split_name, data):
    print(f"\n--- {split_name} Target Distribution (LOAN_APPROVAL) ---")
    counts = data["LOAN_APPROVAL"].value_counts().sort_index()
    percentages = (data["LOAN_APPROVAL"].value_counts(normalize=True).sort_index() * 100).round(2)
    dist_df = pd.DataFrame({"Count": counts, "Percentage (%)": percentages})
    print(dist_df)

# Print distributions for train, validation, and test sets
print_target_distribution("TRAINING SET", train_df)
print_target_distribution("VALIDATION SET", val_df)
print_target_distribution("TEST SET", test_df)

# 2. List all CSV files in data/processed/
print("\n" + "=" * 45)
print("     SAVED FILES IN data/processed/")
print("=" * 45)

csv_files = sorted(list(processed_dir.glob("*.csv")))
for file_path in csv_files:
    file_size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"- {file_path.name:<20} ({file_size_mb:.2f} MB)")


--- TRAINING SET Target Distribution (LOAN_APPROVAL) ---
               Count  Percentage (%)
LOAN_APPROVAL                       
0              18940           27.06
1              51059           72.94

--- VALIDATION SET Target Distribution (LOAN_APPROVAL) ---
               Count  Percentage (%)
LOAN_APPROVAL                       
0               4058           27.06
1              10941           72.94

--- TEST SET Target Distribution (LOAN_APPROVAL) ---
               Count  Percentage (%)
LOAN_APPROVAL                       
0               4060           27.06
1              10942           72.94

     SAVED FILES IN data/processed/
- cleaned_data.csv     (7.73 MB)
- test.csv             (1.16 MB)
- train.csv            (5.41 MB)
- validation.csv       (1.16 MB)


In [46]:
from pathlib import Path
import numpy as np
import pandas as pd

# Safe path resolution that works in both .py files and interactive notebooks
if "__file__" in globals():
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
else:
    PROJECT_ROOT = (
        Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
    )

CLEANED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_data.csv"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_FILE = OUTPUT_DIR / "generated_water_quality_test_scenarios.csv"

RANDOM_SEED = 42
TARGET_COL = "LOAN_APPROVAL"


def generate_scenarios(
    base_df: pd.DataFrame, num_scenarios: int = 256, seed: int = RANDOM_SEED
) -> pd.DataFrame:
    """Generates synthetic stress-test scenarios (edge cases/perturbations)

    derived from the cleaned baseline data.
    """
    np.random.seed(seed)

    sample_df = base_df.sample(
        n=num_scenarios, replace=True, random_state=seed
    ).copy()

    numeric_cols = sample_df.select_dtypes(include="number").columns.tolist()
    if TARGET_COL in numeric_cols:
        numeric_cols.remove(TARGET_COL)

    for col in numeric_cols:
        std = sample_df[col].std()
        jitter = np.random.normal(
            0, 0.05 * (std if std > 0 else 1.0), size=num_scenarios
        )
        sample_df[col] = sample_df[col] + jitter

        if col in ["age", "duration", "campaign", "previous"]:
            sample_df[col] = sample_df[col].clip(lower=0)
        if col == "age":
            sample_df[col] = sample_df[col].clip(lower=18, upper=120)

    int_cols = ["age", "day", "duration", "campaign", "pdays", "previous"]
    for col in int_cols:
        if col in sample_df.columns:
            sample_df[col] = sample_df[col].round().astype(int)

    return sample_df.reset_index(drop=True)


def run_generator() -> None:
    """Orchestrates test scenario creation and saves the CSV artifact."""
    if not CLEANED_DATA_PATH.exists():
        raw_path = (
            PROJECT_ROOT
            / "data"
            / "raw"
            / "banking_loan_approval_dataset_100k.csv"
        )
        base_df = pd.read_csv(raw_path, encoding="utf-8-sig")
    else:
        base_df = pd.read_csv(CLEANED_DATA_PATH)

    base_df.columns = base_df.columns.str.strip()

    scenarios_df = generate_scenarios(
        base_df, num_scenarios=256, seed=RANDOM_SEED
    )

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    scenarios_df.to_csv(OUTPUT_FILE, index=False)
    print(
        f"Successfully generated {len(scenarios_df)} scenarios at: {OUTPUT_FILE}"
    )


if __name__ == "__main__":
    run_generator()

Successfully generated 256 scenarios at: c:\Users\ayush\OneDrive\Desktop\Project26-27\AHPV_Credit_Risk_Detection_Model\data\processed\generated_water_quality_test_scenarios.csv


In [49]:
from pathlib import Path
import pandas as pd

# 1. Resolve project root
PROJECT_ROOT = (
    Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
)

# 2. Path to the banking credit risk stress scenarios
scenarios_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "generated_loan_approval_test_scenarios.csv"
)

# Fallback check if the file was saved with the generic autograder name
if not scenarios_path.exists():
    scenarios_path = (
        PROJECT_ROOT
        / "data"
        / "processed"
        / "generated_water_quality_test_scenarios.csv"
    )

scenario_df = pd.read_csv(scenarios_path)

# 3. Terminal Inspection Display
print("=" * 60)
print("   SC09 BANKING STRESS-TEST SCENARIOS INSPECTION")
print("=" * 60)
print(f"Artifact File Path : {scenarios_path.relative_to(PROJECT_ROOT)}")
print(
    f"Dataset Dimensions : {scenario_df.shape[0]} rows, {scenario_df.shape[1]} columns"
)

# 4. Target distribution for LOAN_APPROVAL (0 = Rejected, 1 = Approved)
target_col = (
    "LOAN_APPROVAL" if "LOAN_APPROVAL" in scenario_df.columns else "Potability"
)

print(f"\n--- Target Class Distribution ({target_col}) ---")
counts = scenario_df[target_col].value_counts().sort_index()
proportions = (
    scenario_df[target_col].value_counts(normalize=True).sort_index() * 100
).round(2)
dist_df = pd.DataFrame(
    {
        "Decision": ["Rejected (0)", "Approved (1)"]
        if len(counts) == 2
        else counts.index,
        "Scenario Count": counts.values,
        "Percentage (%)": proportions.values,
    }
)
print(dist_df.to_string(index=False))

print("\n--- First Five Generated Banking Profiles ---")
scenario_df.head()

   SC09 BANKING STRESS-TEST SCENARIOS INSPECTION
Artifact File Path : data\processed\generated_water_quality_test_scenarios.csv
Dataset Dimensions : 256 rows, 17 columns

--- Target Class Distribution (LOAN_APPROVAL) ---
    Decision  Scenario Count  Percentage (%)
Rejected (0)              80           31.25
Approved (1)             176           68.75

--- First Five Generated Banking Profiles ---


,age,job,marital,education,default,balance,housing_loan,personal_loan,contact,day,month,duration,campaign,pdays,previous,poutcome,LOAN_APPROVAL
0,31,technician,divorced,secondary,0.017033,63150.306469,-0.015657,0.042843,cellular,6,aug,1006,3,468,0,unknown,1
1,45,services,married,secondary,-0.009514,94107.146743,1.045261,1.010821,telephone,22,jun,1933,6,377,5,success,0
2,34,management,divorced,secondary,0.005967,-828.639967,0.017674,0.000942,cellular,24,may,317,6,377,9,failure,0
3,50,admin,divorced,secondary,0.010415,150649.337323,0.985954,1.002977,cellular,18,sep,6,16,387,4,unknown,0
4,30,admin,married,tertiary,-0.012462,130380.901315,1.015793,0.015215,cellular,9,jul,195,9,15,3,unknown,1
